## Experimentation pour encodage efficace one hot a partir du data explore

Le data preprocesse sera sauvegarde en format hdf5. L'encodate onehot ne sera probablement pas sauvegarde par sample en tant que tel (perte d'espace disque). Idem pour les chaines de caracteres **primary_label** et **common_name**. On va passer par une indirection. Ce notebook explore le processus pour construire le lookup.

In [1]:
#
# assure le reload de src si modifications sont faite
#
%load_ext autoreload
%autoreload 2

In [2]:
#
# import utilitaires
#
%matplotlib inline

import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from pathlib import Path

In [3]:
#
# import package develope pour le projet
#
import ffury
from ffury.configs import (
    DatasetType,
    DEFAULT_CONFIG_FILE,
    load_config
)

def get_config():
    # creation config - on sait que DEFAULT_CONFIG_FILE est dans le repertoire parent
    config = Path(ffury.__file__).parents[2].joinpath(DEFAULT_CONFIG_FILE)
    return load_config(config)

config = get_config()

# load dataset BirdCLEF
data_df = pd.read_csv(config.get_csv_filename(DatasetType.EXPLORED))

display(data_df.head())
print(data_df.shape)

,common_name,primary_label,latitude,longitude,filename
0,African Bare-eyed Thrush,abethr1,4.3906,38.2788,abethr1/XC128013.ogg
1,African Bare-eyed Thrush,abethr1,-2.9524,38.2921,abethr1/XC363501.ogg
2,African Bare-eyed Thrush,abethr1,-2.9524,38.2921,abethr1/XC363502.ogg
3,African Bare-eyed Thrush,abethr1,-2.9524,38.2921,abethr1/XC363504.ogg
4,African Bare-eyed Thrush,abethr1,-2.9524,38.2921,abethr1/XC379322.ogg


(14842, 5)


In [4]:
species_lookup = data_df[["primary_label", "common_name"]].groupby("primary_label").first()
species_lookup.reset_index(inplace=True)

num_species = species_lookup.shape[0]
eye = np.eye(num_species, dtype=np.uint8)
species_lookup["ohe"] = [eye[i]   for i in range(num_species)]

indices = pd.Categorical(data_df["primary_label"], categories=species_lookup["primary_label"])

if "primary_label_index" in data_df.columns:
    data_df.drop(columns=["primary_label_index"], inplace=True)

data_df.insert(loc=2, column="primary_label_index", value=indices.codes)


display(species_lookup.head())
print(species_lookup.shape)

,primary_label,common_name,ohe
0,abethr1,African Bare-eyed Thrush,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,abhori1,African Black-headed Oriole,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,abythr1,Abyssinian Thrush,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,afbfly1,African Blue Flycatcher,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,afdfly1,African Dusky Flycatcher,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


(212, 3)


In [5]:
display(data_df.head())
print(data_df.shape)

for _, r in data_df.iterrows():
    names = species_lookup.loc[ r["primary_label_index"] ]
    assert r["common_name"] == names["common_name"] and r["primary_label"] == names["primary_label"]

,common_name,primary_label,primary_label_index,latitude,longitude,filename
0,African Bare-eyed Thrush,abethr1,0,4.3906,38.2788,abethr1/XC128013.ogg
1,African Bare-eyed Thrush,abethr1,0,-2.9524,38.2921,abethr1/XC363501.ogg
2,African Bare-eyed Thrush,abethr1,0,-2.9524,38.2921,abethr1/XC363502.ogg
3,African Bare-eyed Thrush,abethr1,0,-2.9524,38.2921,abethr1/XC363504.ogg
4,African Bare-eyed Thrush,abethr1,0,-2.9524,38.2921,abethr1/XC379322.ogg


(14842, 6)
